# Test de 3 modeles de regression

Ce notebook compare trois modeles pour predire `Energy_Consumption_kWh` :

- baseline : regression lineaire avec standardisation ;
- modele avance : Random Forest Regressor ;
- modele optimise : Random Forest Regressor avec Optuna.

Les modeles finaux sont sauvegardes dans `models/` et les metriques dans `results/test_models_metrics.csv`.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_TRIALS = 20

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import load_dataset_split
from metrics import compute_metrics

## Chargement du dataset

On utilise le meme split que le reste du projet pour garder une comparaison juste entre les modeles.

In [2]:
X_train, X_test, y_train, y_test = load_dataset_split()

X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
display(X_train.head())

Train: (72000, 6)
Validation: (14400, 6)
Test: (18000, 6)


,Household_Size,Avg_Temperature_C,Has_AC_Binary,Peak_Hours_Usage_kWh,temperature_x_ac,household_size_x_ac
0,4,19.8,1,6.5,19.8,4
1,2,13.2,1,3.3,13.2,2
2,3,15.4,1,5.7,15.4,3
3,4,19.2,0,2.8,0.0,0
4,6,17.6,0,6.4,0.0,0


In [3]:
results = []


def evaluate_model(model_name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    metrics = compute_metrics(y_eval, y_pred)
    row = {"model": model_name, **metrics}
    results.append(row)
    return row

## Modele 1 - baseline

La baseline est une regression lineaire simple, standardisee avec `StandardScaler`.

In [4]:
baseline_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

baseline_model.fit(X_train, y_train)
evaluate_model("baseline_linear_regression", baseline_model, X_test, y_test)

{'model': 'baseline_linear_regression',
 'mae': 0.6097978740544798,
 'mse': 0.5853825170304991,
 'rmse': 0.7651029453808809,
 'r2': 0.9808082943093896}

## Modele 2 - Random Forest

Ce modele sert de modele avance non optimise.

In [5]:
random_forest_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=16,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_forest_model.fit(X_train, y_train)
evaluate_model("random_forest", random_forest_model, X_test, y_test)

{'model': 'random_forest',
 'mae': 0.5036189048158214,
 'mse': 0.45818272998688025,
 'rmse': 0.6768919633049872,
 'r2': 0.9849785262617085}

## Modele 3 - Random Forest optimise avec Optuna

Optuna cherche les meilleurs hyperparametres sur un split de validation, puis le meilleur modele est re-entraine sur tout le train set.

In [7]:
try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Optuna n'est pas installe. Installe-le avec: pip install optuna"
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [8]:
sample_size = min(20000, len(X_train_model))
X_optuna_train = X_train_model.sample(n=sample_size, random_state=RANDOM_STATE)
y_optuna_train = y_train_model.loc[X_optuna_train.index]


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 250),
        "max_depth": trial.suggest_int("max_depth", 4, 24),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 1.0]
        ),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }

    model = RandomForestRegressor(**params)
    model.fit(X_optuna_train, y_optuna_train)
    y_val_pred = model.predict(X_val)
    return compute_metrics(y_val, y_val_pred)["rmse"]


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

  0%|          | 0/20 [00:00<?, ?it/s]

Best RMSE: 0.6454548744805633
Best params: {'n_estimators': 199, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 1.0}


In [9]:
optuna_rf_model = RandomForestRegressor(
    **study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

optuna_rf_model.fit(X_train, y_train)
evaluate_model("random_forest_optuna", optuna_rf_model, X_test, y_test)

{'model': 'random_forest_optuna',
 'mae': 0.4875417433046164,
 'mse': 0.4229722991633689,
 'rmse': 0.6503632055731389,
 'r2': 0.9861328966194574}

## Comparaison et sauvegarde

On sauvegarde les metriques et les trois modeles entraines.

In [10]:
comparison_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

comparison_df.to_csv(RESULTS_DIR / "test_models_metrics.csv", index=False)

joblib.dump(baseline_model, MODELS_DIR / "baseline_linear_regression.joblib")
joblib.dump(random_forest_model, MODELS_DIR / "random_forest.joblib")
joblib.dump(optuna_rf_model, MODELS_DIR / "random_forest_optuna.joblib")

display(comparison_df)

,model,mae,mse,rmse,r2
0,random_forest_optuna,0.487542,0.422972,0.650363,0.986133
1,random_forest,0.503619,0.458183,0.676892,0.984979
2,baseline_linear_regression,0.609798,0.585383,0.765103,0.980808
